# 2048 AI - Treino DQN no Google Colab

Este notebook valida o primeiro algoritmo de aprendizagem do projeto: um DQN basico. O treino inicial e curto para confirmar que o pipeline funciona antes de gastar tempo com treino longo.

## 1. Preparar ambiente

Use CPU para validacao curta. GPU pode ser ativada depois em `Ambiente de execucao > Alterar tipo de ambiente de execucao`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Sankofa-JBC/2048-ai.git"
PROJECT_DIR = Path("/content/2048-ai")
SRC_DIR = PROJECT_DIR / "src"

if PROJECT_DIR.exists():
    print("Repositorio ja existe. Atualizando...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)
else:
    print("Clonando repositorio...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

if not (PROJECT_DIR / "pyproject.toml").exists():
    raise FileNotFoundError("pyproject.toml nao encontrado na raiz do projeto clonado.")

print("Instalando projeto e dependencias de aprendizagem...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[learning]"],
    cwd=PROJECT_DIR,
    check=True,
)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Raiz do projeto:", PROJECT_DIR)

## 2. Conferir PyTorch e imports

Se esta celula falhar, o ambiente ainda nao esta pronto para treino.

In [ ]:
import torch
from game2048.learning.training import DQNTrainingConfig, train_dqn

print("Torch:", torch.__version__)
print("CUDA disponivel:", torch.cuda.is_available())

## 3. Rodar testes

Antes do treino, confirmamos que o jogo, agentes e avaliador continuam funcionando no Colab.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-t", "."],
    cwd=PROJECT_DIR,
    check=True,
)

## 4. Treino curto de validacao

Este treino nao tem objetivo de jogar bem ainda. Ele serve para validar loop, replay memory, modelo e salvamento de checkpoint.

In [ ]:
subprocess.run([
    sys.executable,
    "train_dqn.py",
    "--episodes",
    "50",
    "--seed",
    "42",
    "--batch-size",
    "32",
    "--min-replay-size",
    "100",
    "--memory-capacity",
    "5000",
    "--output",
    "models/dqn_smoke.pt",
], cwd=PROJECT_DIR, check=True)

## 5. Avaliar o checkpoint DQN

Agora avaliamos o modelo salvo. Nesta fase inicial, o DQN pode ficar pior que o heuristico; o objetivo aqui e validar funcionamento.

In [ ]:
subprocess.run([
    sys.executable,
    "evaluate_dqn.py",
    "models/dqn_smoke.pt",
    "--games",
    "20",
    "--seed",
    "42",
    "--output",
    "results/dqn_smoke.json",
], cwd=PROJECT_DIR, check=True)

## 6. Comparar com baselines

Comparamos rapidamente contra random e heuristic para ter contexto.

In [ ]:
subprocess.run([sys.executable, "evaluate_agents.py", "--agent", "random", "--games", "20", "--seed", "42"], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, "evaluate_agents.py", "--agent", "heuristic", "--games", "20", "--seed", "42"], cwd=PROJECT_DIR, check=True)

## 7. Baixar checkpoint opcionalmente

Use esta celula quando quiser baixar o modelo treinado para o PC.

In [ ]:
# Descomente as duas linhas abaixo no Colab para baixar o checkpoint.
# from google.colab import files
# files.download(str(PROJECT_DIR / "models/dqn_smoke.pt"))

## Proximo passo

Se esse treino curto funcionar, podemos aumentar episodios, ajustar recompensa e comparar a curva de desempenho.